# Data Manipulation with Pandas — French public datasets

This notebook follows the Week 9 lesson on **concatenation, merging, grouping and reshaping**, using the real datasets introduced in Week 8.

## Datasets used
- 2022 first-round presidential election results by département
- 2026 national register of elected officials (mayors and deputies)
- World Bank indicator: women in national parliaments, plus country metadata

The data are downloaded from the course repository into `data/`. The Week 8 cleaning steps are repeated here so this notebook can run independently.

**Important:** the elected-officials files contain personal information. We immediately discard names and dates of birth; the exercises use only aggregate and administrative fields.

In [1]:
import os
import urllib.request
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 20)

BASE_URL = "https://raw.githubusercontent.com/cg67qowo/Introduction-to-Python/main/Week08-ImportingCleaning/data/"
FILES = [
    "resultats_t1_departements_2022.txt",
    "elus_maires_2026.csv",
    "elus_deputes_2026.csv",
    "wb_women_parliament.csv",
    "wb_country_metadata.csv",
    "departements.json",
]
os.makedirs("data", exist_ok=True)
for name in FILES:
    target = os.path.join("data", name)
    if not os.path.exists(target):
        urllib.request.urlretrieve(BASE_URL + name, target)

print("pandas:", pd.__version__)
print("Files:", sorted(os.listdir("data")))

pandas: 3.1.1
Files: ['departements.json', 'elus_deputes_2026.csv', 'elus_maires_2026.csv', 'resultats_t1_departements_2022.txt', 'wb_country_metadata.csv', 'wb_women_parliament.csv']


## 1. Prepare the Week 8 datasets

### 1.1 Election results

The text file is Latin-1 encoded, uses semicolons and French decimal commas. Its header is incomplete, so we reconstruct the 89 column names.

In [2]:
path = "data/resultats_t1_departements_2022.txt"
with open(path, encoding="latin-1") as f:
    header = f.readline()

base_columns = header.strip().split(";")[:17]
candidates = [
    "ARTHAUD", "ROUSSEL", "MACRON", "LASSALLE", "LE PEN", "ZEMMOUR",
    "MELENCHON", "HIDALGO", "JADOT", "PECRESSE", "POUTOU", "DUPONT-AIGNAN"
]
column_names = list(base_columns)
for candidate in candidates:
    for field in ["sexe", "nom", "prenom", "voix", "pct_ins", "pct_exp"]:
        column_names.append(field + "_" + candidate)

election = pd.read_csv(
    path, encoding="latin-1", sep=";", decimal=",",
    names=column_names, header=0, index_col=False,
    dtype={"Code du département": str},
)
print(election.shape)
election.head(3)

(107, 89)


,Code du département,Libellé du département,Etat saisie,Inscrits,Abstentions,% Abs/Ins,Votants,...,pct_exp_POUTOU,sexe_DUPONT-AIGNAN,nom_DUPONT-AIGNAN,prenom_DUPONT-AIGNAN,voix_DUPONT-AIGNAN,pct_ins_DUPONT-AIGNAN,pct_exp_DUPONT-AIGNAN
0,01,Ain,Complet,438109,97541,22.26,340568,...,0.65,M,DUPONT-AIGNAN,Nicolas,8998,2.05,2.70
1,02,Aisne,Complet,373544,101089,27.06,272455,...,0.80,M,DUPONT-AIGNAN,Nicolas,5790,1.55,2.18
2,03,Allier,Complet,249991,58497,23.40,191494,...,0.81,M,DUPONT-AIGNAN,Nicolas,4216,1.69,2.27


### 1.2 Mayors and deputies

We remove names and birth dates, then standardise the administrative column names. This makes later joins easier to read.

In [3]:
def load_elected(path):
    raw = pd.read_csv(path, sep=";")
    private_columns = ["Nom de l'élu", "Prénom de l'élu", "Date de naissance"]
    clean = raw.drop(columns=private_columns, errors="ignore")
    return clean.rename(columns={
        "Code du département": "dept_code",
        "Libellé du département": "dept",
        "Code de la collectivité à statut particulier": "coll_code",
        "Libellé de la collectivité à statut particulier": "coll",
        "Code de la commune": "commune_code",
        "Libellé de la commune": "commune",
        "Code sexe": "sex",
        "Code de la catégorie socio-professionnelle": "job_code",
        "Libellé de la catégorie socio-professionnelle": "job",
        "Date de début du mandat": "mandate_start",
        "Date de début de la fonction": "function_start",
    })

mayors = load_elected("data/elus_maires_2026.csv")
deputies = load_elected("data/elus_deputes_2026.csv")
print("Mayors:", mayors.shape, "| Deputies:", deputies.shape)
mayors.head(3)

Mayors: (34826, 11) | Deputies: (577, 10)


,dept_code,dept,coll_code,coll,commune_code,commune,sex,job_code,job,mandate_start,function_start
0,01,Ain,NaN,NaN,01001,L'Abergement-Clémenciat,F,38,Ingénieur et cadre technique d'entreprise,2026-03-15,2026-03-23
1,01,Ain,NaN,NaN,01002,L'Abergement-De-Varey,M,31,Profession libérale,2026-03-15,2026-03-20
2,01,Ain,NaN,NaN,01004,Ambérieu-En-Bugey,M,74,Ancien cadre,2026-03-15,2026-03-20


### 1.3 World Bank data

The indicator file has metadata lines before the actual header. Country metadata provides the region and income group used later.

In [4]:
wb_raw = pd.read_csv("data/wb_women_parliament.csv", skiprows=4)
meta = pd.read_csv("data/wb_country_metadata.csv").dropna(axis=1, how="all")

# Keep actual country rows: World Bank aggregate groups have no region metadata.
year_columns = [str(y) for y in range(1990, 2026) if str(y) in wb_raw.columns]
countries = wb_raw[["Country Name", "Country Code"] + year_columns].copy()
countries = countries.rename(columns={"Country Name": "country", "Country Code": "country_code"})
countries.head()

,country,country_code,1990,1991,1992,1993,1994,...,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,NaN,NaN,NaN,NaN,NaN,...,29.274564,30.128447,30.474788,30.523627,30.518059,30.232558,30.424571
2,Afghanistan,AFG,NaN,NaN,NaN,NaN,NaN,...,27.868852,27.016129,27.016129,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,NaN,NaN,NaN,NaN,NaN,...,15.398773,16.345271,17.824377,19.182782,20.767717,20.784835,21.120554
4,Angola,AGO,NaN,NaN,NaN,NaN,NaN,...,30.000000,30.000000,29.545455,33.636364,33.636364,39.545455,39.090909


## 2. Concatenation

`pd.concat()` stacks tables along rows (`axis=0`) or aligns them side-by-side (`axis=1`). Here, mayors and deputies share a broadly similar administrative schema, so they can be stacked after adding a role label.

In [5]:
mayors_tagged = mayors.assign(role="Mayor")
deputies_tagged = deputies.assign(role="Deputy")
elected = pd.concat([mayors_tagged, deputies_tagged], ignore_index=True, sort=False)
print("Mayors:", len(mayors), "Deputies:", len(deputies), "Combined:", len(elected))
elected[["role", "dept_code", "dept", "commune_code", "commune"]].head()

Mayors: 34826 Deputies: 577 Combined: 35403


,role,dept_code,dept,commune_code,commune
0,Mayor,01,Ain,01001,L'Abergement-Clémenciat
1,Mayor,01,Ain,01002,L'Abergement-De-Varey
2,Mayor,01,Ain,01004,Ambérieu-En-Bugey
3,Mayor,01,Ain,01005,Ambérieux-En-Dombes
4,Mayor,01,Ain,01006,Ambléon


In [6]:
# keys preserve the source table as an outer index level
elected_by_source = pd.concat(
    [mayors, deputies], keys=["mayors", "deputies"], names=["source", "row"]
)
elected_by_source.head()

dept_code dept coll_code coll commune_code                  commune sex  job_code  \
source row                                                                                     
mayors 0          01  Ain       NaN  NaN        01001  L'Abergement-Clémenciat   F        38   
       1          01  Ain       NaN  NaN        01002    L'Abergement-De-Varey   M        31   
       2          01  Ain       NaN  NaN        01004        Ambérieu-En-Bugey   M        74   
       3          01  Ain       NaN  NaN        01005      Ambérieux-En-Dombes   F        21   
       4          01  Ain       NaN  NaN        01006                  Ambléon   M        22   

                                                  job mandate_start function_start  \
source row                                                                           
mayors 0    Ingénieur et cadre technique d'entreprise    2026-03-15     2026-03-23   
       1                          Profession libérale    2026-03-15     2026-03-20   
       2                                 Ancien cadre    2026-03-15     2026-03-20   
       3                                      Artisan    2026-03-15     2026-03-20   
       4                       Commerçant et assimilé    2026-03-15     2026-03-20   

           Code de la circonscription législative Libellé de la circonscription législative  
source row                                                                                   
mayors 0                                      NaN                                       NaN  
       1                                      NaN                                       NaN  
       2                                      NaN                                       NaN  
       3                                      NaN                                       NaN  
       4                                      NaN                                       NaN

**Exercise 2.1.** Create a DataFrame containing only mayors and deputies from département 75. Keep the original index, then repeat with `ignore_index=True`. Compare the indexes.

In [7]:
# Your code here

## 3. Merge and join

A merge combines rows using an explicit key. The World Bank country code is the key linking indicator values to country metadata. Use `validate` to check the expected relationship.

In [9]:
meta_small = meta[["Country Code", "Region", "IncomeGroup", "TableName"]].rename(
    columns={"Country Code": "country_code", "TableName": "country"}
)
country_panel = countries.merge(
    meta_small, on="country_code", how="left", validate="one_to_one",
    suffixes=("", "_metadata")
)
country_panel[["country", "country_code", "Region", "IncomeGroup", "2025"]].head()

,country,country_code,Region,IncomeGroup,2025
0,Aruba,ABW,Latin America & Caribbean,High income,NaN
1,Africa Eastern and Southern,AFE,NaN,NaN,30.424571
2,Afghanistan,AFG,Middle East & North Africa,Low income,NaN
3,Africa Western and Central,AFW,NaN,NaN,21.120554
4,Angola,AGO,Sub-Saharan Africa,Lower middle income,39.090909


In [10]:
# Compare join types: left keeps every indicator row; inner keeps only matched country codes.
left = countries.merge(meta_small, on="country_code", how="left")
inner = countries.merge(meta_small, on="country_code", how="inner")
print("Indicator rows:", len(countries))
print("Left join rows:", len(left))
print("Inner join rows:", len(inner))
print("Rows without metadata:", left["Region"].isna().sum())

Indicator rows: 266
Left join rows: 266
Inner join rows: 265
Rows without metadata: 49


### 3.1 Merge election results with département labels

The Week 8 `departements.json` file provides reference labels. We normalise its structure before merging; inspect the first rows because JSON layouts can differ.

In [11]:
departements = pd.read_json("data/departements.json")
departements.head()

,nom,code,codeRegion
0,Ain,01,84
1,Aisne,02,32
2,Allier,03,84
3,Alpes-de-Haute-Provence,04,93
4,Hautes-Alpes,05,93


In [12]:
# Inspect columns and choose the code/label fields present in this version of the reference file.
print(departements.columns.tolist())

['nom', 'code', 'codeRegion']


**Exercise 3.1.** Identify the département code and name columns in `departements`. Rename them to `dept_code` and `dept_name`, then left-merge them onto `election` using the département code. Check how many rows have no matching label.

In [13]:
# Your code here

In [14]:
print(election.columns.tolist())

['Code du département', 'Libellé du département', 'Etat saisie', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', 'sexe_ARTHAUD', 'nom_ARTHAUD', 'prenom_ARTHAUD', 'voix_ARTHAUD', 'pct_ins_ARTHAUD', 'pct_exp_ARTHAUD', 'sexe_ROUSSEL', 'nom_ROUSSEL', 'prenom_ROUSSEL', 'voix_ROUSSEL', 'pct_ins_ROUSSEL', 'pct_exp_ROUSSEL', 'sexe_MACRON', 'nom_MACRON', 'prenom_MACRON', 'voix_MACRON', 'pct_ins_MACRON', 'pct_exp_MACRON', 'sexe_LASSALLE', 'nom_LASSALLE', 'prenom_LASSALLE', 'voix_LASSALLE', 'pct_ins_LASSALLE', 'pct_exp_LASSALLE', 'sexe_LE PEN', 'nom_LE PEN', 'prenom_LE PEN', 'voix_LE PEN', 'pct_ins_LE PEN', 'pct_exp_LE PEN', 'sexe_ZEMMOUR', 'nom_ZEMMOUR', 'prenom_ZEMMOUR', 'voix_ZEMMOUR', 'pct_ins_ZEMMOUR', 'pct_exp_ZEMMOUR', 'sexe_MELENCHON', 'nom_MELENCHON', 'prenom_MELENCHON', 'voix_MELENCHON', 'pct_ins_MELENCHON', 'pct_exp_MELENCHON', 'sexe_HIDALGO', 'nom_HIDALGO

## 4. GroupBy — split, apply, combine

`groupby()` splits rows into groups, applies an aggregation, and combines the results. We will calculate representation statistics by département and by socio-professional category.

In [16]:
# Share of women among mayors by socio-professional category.
mayors_analysis = mayors.copy()
mayors_analysis["is_woman"] = mayors_analysis["sex"].astype("string").str.upper().eq("F")
mayors_analysis.groupby("job")["is_woman"].mean().mul(100).sort_values(ascending=False).round(1).head(12)

job
Personnel des services directs aux particuliers                                      83.1
Employé administratif d'entreprise                                                   76.2
Profession intermédiaire de la santé et du travail social                            60.4
Personne diverse sans activité professionnelle de 60 ans et plus (sauf retraité)     56.2
Profession intermédiaire administrative de la fonction publique                      55.0
Personne diverse sans activité professionnelle de moins de 60 ans (sauf retraité)    51.0
Profession intermédiaire administrative et commerciale des entreprises               47.6
Professeur des écoles, instituteur et assimilé                                       44.2
Employé de commerce                                                                  39.4
Employé civil et agent de service de la fonction publique                            33.6
Ancienne profession intermédiaire                                                    31.9
Ancien

In [17]:
# Multiple statistics for each département.
dept_mayors = mayors.groupby("dept_code").agg(
    n_mayors=("commune_code", "size"),
    n_communes=("commune_code", "nunique"),
    n_jobs=("job", "nunique"),
).sort_values("n_mayors", ascending=False)
dept_mayors.head(10)

,n_mayors,n_communes,n_jobs
dept_code,,,
62,885,885,39
02,793,793,39
80,770,770,39
57,721,721,38
76,707,707,35
21,698,698,37
60,678,678,38
59,647,647,35
51,610,610,38


In [18]:
# Aggregate election results by département: votes and turnout/abstention.
election_summary = election.groupby("Code du département", as_index=False).agg(
    registered=("Inscrits", "sum"),
    abstention_rate=("% Abs/Ins", "mean"),
    macron_votes=("voix_MACRON", "sum"),
    le_pen_votes=("voix_LE PEN", "sum"),
)
election_summary.sort_values("macron_votes", ascending=False).head()

,Code du département,registered,abstention_rate,macron_votes,le_pen_votes
75,75,1368025,21.69,372820,58429
59,59,1813906,28.34,336138,373127
92,92,997667,20.93,287494,64812
69,69,1181226,21.64,278243,150463
44,44,1056693,22.37,256609,135702


**Exercise 4.1.** Find the five départements with the highest mean abstention rate, excluding overseas codes beginning with `Z`. Display the département code and abstention rate.

**Exercise 4.2.** Calculate the number of mayors and the share of women by département. Sort by the share of women.

In [19]:
# Your code here

### 4.1 Grouping World Bank indicator values

The `Region` and `IncomeGroup` columns allow comparisons across groups of countries. First convert the year columns to numeric and use only observations from the same year.

In [22]:
for year in year_columns:
    country_panel[year] = pd.to_numeric(country_panel[year], errors="coerce")

country_panel.groupby("Region")["2025"].agg(["count", "mean", "median"]).round(1)

,count,mean,median
Region,,,
East Asia & Pacific,29,18.6,17.6
Europe & Central Asia,52,32.3,33.4
Latin America & Caribbean,32,31.5,31.3
Middle East & North Africa,20,17.2,18.3
North America,2,29.6,29.6
South Asia,4,7.8,7.0
Sub-Saharan Africa,45,25.8,23.4


In [23]:
country_panel.groupby("IncomeGroup").agg(
    countries_with_data=("2025", "count"),
    average=("2025", "mean"),
    median=("2025", "median")
).round(1).sort_values("average", ascending=False)

,countries_with_data,average,median
IncomeGroup,,,
High income,63,30.2,31.3
Upper middle income,58,24.7,22.8
Low income,21,23.9,19.6
Lower middle income,42,23.3,23.3


**Exercise 4.3.** Compare the 2000 and 2025 averages by income group. How many non-missing observations are available in each year? Explain why averages based on different sets of countries should be interpreted carefully.

In [24]:
# Your code here

## 5. Reshaping data

Wide data has one column per year. Long data has one row per country-year. `melt()` converts wide to long; `pivot_table()` converts long data back into a summary table.

In [26]:
# Wide -> long
id_vars = ["country", "country_code", "Region", "IncomeGroup"]
wb_long = country_panel.melt(
    id_vars=id_vars, value_vars=year_columns,
    var_name="year", value_name="women_in_parliament_pct"
)
wb_long["year"] = pd.to_numeric(wb_long["year"], errors="coerce")
wb_long.head()

,country,country_code,Region,IncomeGroup,year,women_in_parliament_pct
0,Aruba,ABW,Latin America & Caribbean,High income,1990,NaN
1,Africa Eastern and Southern,AFE,NaN,NaN,1990,NaN
2,Afghanistan,AFG,Middle East & North Africa,Low income,1990,NaN
3,Africa Western and Central,AFW,NaN,NaN,1990,NaN
4,Angola,AGO,Sub-Saharan Africa,Lower middle income,1990,NaN


In [27]:
# Keep years with observed values, then summarise the regional average by year.
regional_trend = (
    wb_long.dropna(subset=["women_in_parliament_pct"])
    .pivot_table(index="year", columns="Region",
                 values="women_in_parliament_pct", aggfunc="mean")
)
regional_trend.tail()

Region,East Asia & Pacific,Europe & Central Asia,Latin America & Caribbean,Middle East & North Africa,North America,South Asia,Sub-Saharan Africa
year,,,,,,,
2021,16.500622,31.250870,29.480177,16.863805,29.061571,15.838167,23.941636
2022,17.161214,31.406674,30.172624,17.495007,29.588538,16.035840,24.368535
2023,17.170691,31.574143,30.645696,16.782562,29.810140,16.084892,25.460427
2024,18.161335,31.815439,31.490539,17.574985,29.695197,12.844499,25.591777
2025,18.623430,32.265943,31.528811,17.245779,29.627942,7.761417,25.771751


In [28]:
# Long -> wide: regional average for selected years.
regional_long = (
    wb_long.dropna(subset=["women_in_parliament_pct"])
    .groupby(["Region", "year"], as_index=False)["women_in_parliament_pct"].mean()
)
regional_wide = regional_long.pivot(
    index="Region", columns="year", values="women_in_parliament_pct"
)
regional_wide[[2000, 2010, 2020, 2025]].round(1)

year,2000,2010,2020,2025
Region,,,,
East Asia & Pacific,9.3,12.1,16.5,18.6
Europe & Central Asia,16.5,22.6,29.8,32.3
Latin America & Caribbean,12.9,18.0,28.1,31.5
Middle East & North Africa,4.2,11.1,17.8,17.2
North America,NaN,19.4,28.2,29.6
South Asia,7.9,13.8,15.5,7.8
Sub-Saharan Africa,10.6,17.6,22.7,25.8


In [29]:
# pivot_table can aggregate duplicate combinations and add totals.
regional_summary = wb_long.pivot_table(
    index="IncomeGroup", columns="year", values="women_in_parliament_pct",
    aggfunc="mean", margins=True, margins_name="All income groups"
)
regional_summary[[2000, 2010, 2020, 2025]].round(1)

year,2000,2010,2020,2025
IncomeGroup,,,,
High income,16.4,20.4,27.1,30.2
Low income,10.8,18.4,21.7,23.9
Lower middle income,7.5,15.4,21.6,23.3
Upper middle income,10.3,15.2,22.8,24.7
All income groups,11.8,17.4,23.8,26.2


**Exercise 5.1.** Create a long-format table containing only the years 2000, 2010, 2020 and 2025. Then create a pivot table with `IncomeGroup` as rows, `year` as columns and the mean indicator value as values.

**Exercise 5.2.** For mayor counts, create a pivot table with département as rows and `role` as columns using the combined `elected` table. (Use `commune_code` as the count value.)

## 6. Stack and unstack

A grouped result with multiple keys has a MultiIndex. `unstack()` moves one index level into columns; `stack()` moves it back.

In [32]:
grouped = mayors.groupby(["dept_code", "sex"]).size()
print(grouped.head(10))
unstacked = grouped.unstack("sex", fill_value=0)
unstacked.head()

dept_code  sex
01         F       98
           M      261
02         F      161
           M      632
03         F       75
           M      241
04         F       46
           M      152
05         F       40
           M      122
dtype: int64


sex,F,M
dept_code,,
01,98,261
02,161,632
03,75,241
04,46,152
05,40,122


In [33]:
restacked = unstacked.stack()
restacked.head(10)

dept_code  sex
01         F       98
           M      261
02         F      161
           M      632
03         F       75
           M      241
04         F       46
           M      152
05         F       40
           M      122
dtype: int64

## 7. Putting the operations together

This final example combines a groupby, a merge and a reshape to create a département-level table of elected-official counts by role.

In [34]:
dept_role = (
    elected.groupby(["dept_code", "role"], as_index=False)
    .agg(number_of_officials=("commune_code", "size"))
)
dept_role_wide = dept_role.pivot(
    index="dept_code", columns="role", values="number_of_officials"
).fillna(0).astype(int)
dept_role_wide.head()

role,Deputy,Mayor
dept_code,,
01,5,359
02,5,793
03,3,316
04,2,198
05,2,162


## 8. Summary

- **`concat`** stacks compatible tables or aligns them along columns.
- **`merge`** combines tables using explicit key columns; inspect unmatched rows and validate key relationships.
- **`groupby`** calculates statistics for categories or combinations of categories.
- **`pivot` / `pivot_table`** turn grouped long data into a wide summary.
- **`melt`** turns wide data into long data.
- **`stack` / `unstack`** move levels between a MultiIndex and columns.

Always check the unit of observation, key uniqueness, missing values and row counts after each operation. A merge that runs without an error is not necessarily a correct merge.